# 04 · Data Inspection and Cleaning

**Goal:** learn the standard first steps for exploring any new dataset, and how to handle
missing values, duplicates, and incorrect dtypes.

### The standard "first look" toolkit

Whenever you load a new dataset, these are the first things to run — practically a ritual in
data analysis.

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana", "Evan", "Alice"],
    "age": [25, 32, np.nan, 47, 29, 25],
    "city": ["NYC", "LA", "Chicago", None, "NYC", "NYC"],
    "score": [88.5, 92.1, 79.3, 85.0, np.nan, 88.5]
})

print(df.head())      # first 5 rows -- your first look at the data

      name   age     city  score
0    Alice  25.0      NYC   88.5
1      Bob  32.0       LA   92.1
2  Charlie   NaN  Chicago   79.3
3    Diana  47.0      NaN   85.0
4     Evan  29.0      NYC    NaN


In [2]:
print(df.head(3))    # first 3 rows (default is 5)
print()
print(df.tail(2))     # last 2 rows

      name   age     city  score
0    Alice  25.0      NYC   88.5
1      Bob  32.0       LA   92.1
2  Charlie   NaN  Chicago   79.3

    name   age city  score
4   Evan  29.0  NYC    NaN
5  Alice  25.0  NYC   88.5


In [3]:
print(df.info())    # column names, dtypes, non-null counts, memory usage -- a great overview

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   name    6 non-null      str    
 1   age     5 non-null      float64
 2   city    5 non-null      str    
 3   score   5 non-null      float64
dtypes: float64(2), str(2)
memory usage: 324.0 bytes
None


In [4]:
print(df.describe())    # statistical summary of NUMERIC columns: count, mean, std, min/max, quartiles

             age      score
count   5.000000   5.000000
mean   31.600000  86.680000
std     9.099451   4.829286
min    25.000000  79.300000
25%    25.000000  85.000000
50%    29.000000  88.500000
75%    32.000000  88.500000
max    47.000000  92.100000


In [5]:
print(df.describe(include="object"))   # summary for non-numeric (string/categorical) columns

         name city
count       6    5
unique      5    3
top     Alice  NYC
freq        2    3


/tmp/ipykernel_826/3515303375.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.describe(include="object"))   # summary for non-numeric (string/categorical) columns


### Shape, dtypes, and column names — the quick facts

In [6]:
print("shape:  ", df.shape)
print("columns:", df.columns.tolist())
print()
print(df.dtypes)

shape:   (6, 4)
columns: ['name', 'age', 'city', 'score']

name         str
age      float64
city         str
score    float64
dtype: object


### Detecting missing values

Missing data shows up as `NaN` (Not a Number) for numeric columns, or `None`/`NaN` for object
columns. `.isnull()` (alias `.isna()`) gives you a boolean mask you can then summarize.

In [7]:
print(df.isnull())          # True/False for every single cell
print()
print(df.isnull().sum())     # count of missing values PER COLUMN -- the most useful summary
print()
print(df.isnull().sum().sum())   # total missing values across the whole DataFrame

    name    age   city  score
0  False  False  False  False
1  False  False  False  False
2  False   True  False  False
3  False  False   True  False
4  False  False  False   True
5  False  False  False  False

name     0
age      1
city     1
score    1
dtype: int64

3


### Handling missing values: drop or fill

There's no single "correct" choice — it depends on the dataset and what the missing value
means. The three most common strategies:

In [8]:
# 1. Drop rows with ANY missing value
print(df.dropna())

    name   age city  score
0  Alice  25.0  NYC   88.5
1    Bob  32.0   LA   92.1
5  Alice  25.0  NYC   88.5


In [9]:
# 2. Drop rows only if ALL values are missing (rare to actually need this)
print(df.dropna(how="all"))

# Drop only if a SPECIFIC column is missing
print(df.dropna(subset=["age"]))

      name   age     city  score
0    Alice  25.0      NYC   88.5
1      Bob  32.0       LA   92.1
2  Charlie   NaN  Chicago   79.3
3    Diana  47.0      NaN   85.0
4     Evan  29.0      NYC    NaN
5    Alice  25.0      NYC   88.5
    name   age city  score
0  Alice  25.0  NYC   88.5
1    Bob  32.0   LA   92.1
3  Diana  47.0  NaN   85.0
4   Evan  29.0  NYC    NaN
5  Alice  25.0  NYC   88.5


In [10]:
# 3. Fill missing values instead of dropping them
df_filled = df.copy()
df_filled["age"] = df_filled["age"].fillna(df_filled["age"].mean())     # fill with column mean
df_filled["city"] = df_filled["city"].fillna("Unknown")                  # fill with a placeholder
df_filled["score"] = df_filled["score"].fillna(df_filled["score"].median())   # fill with median

print(df_filled)
print()
print(df_filled.isnull().sum())    # confirm no more missing values

      name   age     city  score
0    Alice  25.0      NYC   88.5
1      Bob  32.0       LA   92.1
2  Charlie  31.6  Chicago   79.3
3    Diana  47.0  Unknown   85.0
4     Evan  29.0      NYC   88.5
5    Alice  25.0      NYC   88.5

name     0
age      0
city     0
score    0
dtype: int64


### Forward-fill and backward-fill

Especially useful for time series data, where a missing value likely means "same as the last
known value".

In [11]:
s = pd.Series([1, np.nan, np.nan, 4, np.nan, 6])
print("original:      ", s.tolist())
print("ffill (forward):", s.ffill().tolist())    # fills with the last valid value going forward
print("bfill (backward):", s.bfill().tolist())    # fills with the next valid value going backward

original:       [1.0, nan, nan, 4.0, nan, 6.0]
ffill (forward): [1.0, 1.0, 1.0, 4.0, 4.0, 6.0]
bfill (backward): [1.0, 4.0, 4.0, 4.0, 6.0, 6.0]


### Detecting and removing duplicates

In [12]:
print(df.duplicated())          # boolean: is this row an exact duplicate of an earlier row?
print()
print(df[df.duplicated()])       # show just the duplicate rows
print()
print(df.drop_duplicates())       # returns a new DataFrame with duplicates removed

0    False
1    False
2    False
3    False
4    False
5     True
dtype: bool

    name   age city  score
5  Alice  25.0  NYC   88.5

      name   age     city  score
0    Alice  25.0      NYC   88.5
1      Bob  32.0       LA   92.1
2  Charlie   NaN  Chicago   79.3
3    Diana  47.0      NaN   85.0
4     Evan  29.0      NYC    NaN


In [13]:
# Check duplicates based on a SUBSET of columns (e.g. same name = duplicate, ignore other cols)
print(df.duplicated(subset=["name"]))
print(df.drop_duplicates(subset=["name"], keep="first"))   # keep the FIRST occurrence

0    False
1    False
2    False
3    False
4    False
5     True
dtype: bool
      name   age     city  score
0    Alice  25.0      NYC   88.5
1      Bob  32.0       LA   92.1
2  Charlie   NaN  Chicago   79.3
3    Diana  47.0      NaN   85.0
4     Evan  29.0      NYC    NaN


### Fixing dtypes

Data loaded from files (CSV especially) often has the wrong dtype — numbers stored as text,
dates stored as strings, etc. `.astype()` converts explicitly.

In [14]:
messy = pd.DataFrame({
    "price": ["10.5", "20.0", "15.75"],     # numbers stored as strings!
    "quantity": [1, 2, 3]
})
print(messy.dtypes)

messy["price"] = messy["price"].astype(float)   # convert to proper numeric dtype
print(messy.dtypes)
print(messy["price"].sum())    # now math works correctly

price         str
quantity    int64
dtype: object
price       float64
quantity      int64
dtype: object
46.25


### Renaming columns and checking value counts

In [15]:
df_renamed = df.rename(columns={"name": "full_name", "city": "location"})
print(df_renamed.columns.tolist())

# .value_counts() -- extremely useful for quickly understanding a categorical column
print(df["city"].value_counts())
print()
print(df["city"].value_counts(dropna=False))   # include NaN in the counts too

['full_name', 'age', 'location', 'score']
city
NYC        3
LA         1
Chicago    1
Name: count, dtype: int64

city
NYC        3
LA         1
Chicago    1
NaN        1
Name: count, dtype: int64


### Unique values

In [16]:
print(df["city"].unique())        # array of distinct values (including NaN if present)
print(df["city"].nunique())        # count of distinct values (excludes NaN by default)

<StringArray>
['NYC', 'LA', 'Chicago', nan]
Length: 4, dtype: str
3


### 🧠 Quick check

1. What's the difference between `.dropna()` and `.dropna(how="all")`?
2. What does `df["col"].fillna(df["col"].mean())` do?
3. How would you remove duplicate rows based only on the `"name"` column, keeping the first
   occurrence?

<details>
<summary>Answers</summary>

1. `.dropna()` drops a row if it has ANY missing value; `.dropna(how="all")` only drops a row
   if EVERY value in it is missing.
2. It replaces missing values in that column with the column's average — a common, simple
   imputation strategy.
3. `df.drop_duplicates(subset=["name"], keep="first")`.
</details>

### ✍️ Practice

1. Load (or construct) a DataFrame with some missing values and duplicates. Run `.info()` and
   `.describe()` on it.
2. Fill missing numeric values with the column median, and missing string values with
   `"Unknown"`.
3. Find and remove exact duplicate rows.
4. Convert a column of numbers stored as strings into a proper numeric dtype and confirm with
   `.dtypes`.

Continue to **`05_filtering_sorting_modifying.ipynb`** next.